In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
import os
import seaborn as sns
from scipy import stats as scipystats
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option('display.max_colwidth', 150)

# ***import data.csv and analysed.csv files saved in script 4*** 


In [2]:
#######################################import data before cca analysis##################################################################

Ccr4pre = pd.read_csv(r"Y:\Analysed Live cell microscopy data\Ccr4_data_for_Felix\20231212\grand dataset 20231212.csv")
Ccr4post = pd.read_csv(r"Y:\dfs_for_final_code\CombinedCcr4Rep2 SCD LCM analysed.csv")

Ccr4pre['growthmedium'] = 'SCD'
Ccr4post['growthmedium'] = 'SCD'



FileNotFoundError: [Errno 2] No such file or directory: 'Y:\\Analysed Live cell microscopy data\\Ccr4_data_for_Felix\\20231212\\grand dataset 20231212.csv'

In [3]:
############################### add strain information ############################################################################################

Ccr4pre['Strain'] = "CCR4DEL"
Ccr4pre['date'] = 12122023


NameError: name 'Ccr4pre' is not defined

# ***keep columns of choice and label incomplete phases***

In [4]:
#  choose columns needed and label incomplete phases 

comprehensive_WTsizeMLR_df = Ccr4pre[['Position', 'Cell_ID', 'frame_i', 'cell_cycle_stage', 'generation_num', 'relative_ID', 'relationship',
                                        'emerg_frame_i', 'division_frame_i', 'cell_vol_fl' , 'date', 'time_seconds',  'is_cell_dead',
                                        'growthmedium',  'Strain']]

labeled_incomplete_cycles = []
import warnings

with warnings.catch_warnings(record=True):
    for med in comprehensive_WTsizeMLR_df['growthmedium'].unique():
        med_df = comprehensive_WTsizeMLR_df[comprehensive_WTsizeMLR_df['growthmedium'] == med]
        for date in med_df['date'].unique():
            date_df = med_df[med_df['date'] == date]
            for pos in date_df['Position'].unique():
                pos_df = date_df[date_df['Position'] == pos]
                pos_last_frame = (pos_df['frame_i'].unique().max())
                # if (med == 'SCD') & (date == 20012021) & (pos == 'Position_1'):
                #     print (pos_last_frame)
                for cell in pos_df['Cell_ID'].unique():
                    cell_df = pos_df[pos_df['Cell_ID'] == cell]
                    cell_last_frame = cell_df['frame_i'].unique().max()
                    cell_last_frame_id = (cell_df[cell_df['frame_i'] == cell_last_frame]).index



                    cell_last_frame_id = cell_last_frame_id.tolist()
                    cell_df['complete_phase'] = True
                    if cell_last_frame == pos_last_frame:
                        #print ("in the loop")
                        lastccstage = cell_df['cell_cycle_stage'].loc[(cell_last_frame_id[0])]
                        lastgen = cell_df['generation_num'].loc[(cell_last_frame_id[0])]

                        cell_df['complete_phase'].loc[(cell_df[(cell_df['cell_cycle_stage'] == lastccstage) & (cell_df['generation_num'] == lastgen)].index)] = False

                    if cell_last_frame != pos_last_frame:
                        #print (date, pos, cell, pos_last_frame, cell_last_frame)
                        lastccstage = cell_df['cell_cycle_stage'].loc[(cell_last_frame_id[0])]
                        lastgen = cell_df['generation_num'].loc[(cell_last_frame_id[0])]

                        cell_df['complete_phase'].loc[(cell_df[(cell_df['cell_cycle_stage'] == lastccstage) & (cell_df['generation_num'] == lastgen)].index)] = False

                    if len(cell_df['frame_i']) != 0:
                        # print (cell_df['frame_i'].unique().min())
                        if ((cell_df['frame_i'].unique().min()) == 0):

                            cell_first_frame_id = (cell_df[cell_df['frame_i'] == 0]).index
                            cell_first_frame_id = cell_first_frame_id.tolist()
                            firstccstage = cell_df['cell_cycle_stage'].loc[(cell_first_frame_id[0])]
                            firstgen = cell_df['generation_num'].loc[(cell_first_frame_id[0])]

                            # if (med == 'SCD') & (date == 20012021) & (pos == 'Position_1') & (cell == 6):
                            #     print(cell_df)
                            #     print (cell_first_frame_id)
                            #     print (firstccstage)
                            #     print (firstgen)

                            cell_df['complete_phase'].loc[(cell_df[(cell_df['cell_cycle_stage'] == firstccstage) & (cell_df['generation_num'] == firstgen)].index)] = False
                            # if (med == 'SCD') & (date == 20012021) & (pos == 'Position_1') & (cell == 6):
                            #     print(cell_df)
                    if len(cell_df['frame_i']) != 0:
                        labeled_incomplete_cycles.append(cell_df)


    comprehensive_WTsizeMLR_df = pd.concat(labeled_incomplete_cycles).reset_index(drop =True)



NameError: name 'Ccr4pre' is not defined

# ***in analysed data, combine rows of G1 and S info per cycle***

In [5]:
################################################# combine rows of G1 and S info per cycle ########################################
new = []
for date in Ccr4post['date'].unique():
    date_df  = Ccr4post[Ccr4post['date'] == date]
    for pos in date_df['Position_n'].unique():
        pos_df = date_df[date_df['Position_n'] == pos]
        for cell in pos_df['Cell_ID'].unique():
            cell_df = pos_df[pos_df['Cell_ID'] == cell]
            for cycle in cell_df['cell_cycle_num'].unique():
                cycle_df =  cell_df[cell_df['cell_cycle_num'] == cycle]
                if len(cycle_df) > 1 :
                    cycle_df = cycle_df.iloc[0].combine_first(cycle_df.iloc[1])
                    cycle_df = pd.DataFrame(cycle_df)
                    cycle_df = cycle_df.T

                new.append(cycle_df)


Ccr4post = pd.concat(new).reset_index(drop = True)


NameError: name 'Ccr4post' is not defined

# ***combine pre-analysis (with frame info) and post-analysis (with phase info) data***

In [6]:
############################################### combine pre and post analysis data #######################################################################

Ccr4post.rename(columns={"Position_n": "Position", "cell_cycle_num": "generation_num"}, inplace = True)

granddataset = Ccr4post.merge(comprehensive_WTsizeMLR_df, how = 'right', on = ['Position', 'Cell_ID', 'date', 'growthmedium', 'Strain', 'generation_num' ])

NameError: name 'Ccr4post' is not defined

In [7]:
########################## reorder columns ############################################################################################

granddataset = granddataset[["generation_num", "Position", "Cell_ID", "growthmedium", "frame_i", "cell_cycle_stage", "relative_ID", "relationship",  "cell_vol_fl", "emerg_frame_i",
                              "division_frame_i", "len_G1",  "date", "Strain", "Birth frame",
                             "Birth vol fl", "last_G1_frame", "size_G1_end", "vol_added_G1", "Div frame", "Div vol fl", "len_S" ,"mother_size_emerg" ,
                             "mother_size_div" ,"vol_added_S_mother", "bud_size_emerg", "bud_size_div", "vol_added_S_bud", "sys_size_emerg", "sys_size_div" ,
                             "vol_added_S_tot" ,"total vol added" ,   "time_seconds" , "is_cell_dead", "complete_phase"]]

NameError: name 'granddataset' is not defined

In [8]:
# remove cell cycles that have no info due to merge


granddataset.drop(granddataset[np.isnan(granddataset['frame_i']) == True].index, inplace=True)

granddataset['lenG1_mins'] = granddataset['len_G1']*3
granddataset['G1_growthrate_flpermin'] = granddataset['vol_added_G1']/granddataset['lenG1_mins']


NameError: name 'granddataset' is not defined

# ***calculating abs growth rate (fl/min) from each cycle's first frame vol to last frame volume***

In [9]:
############################################ calculating abs growth rate from each cycle's first frame vol to last frame volume ##############################################

new = []
import warnings

with warnings.catch_warnings(record=True):
    for date in granddataset['date'].unique():
        date_df  = granddataset[granddataset['date'] == date]
        for pos in date_df['Position'].unique():
            pos_df = date_df[date_df['Position'] == pos]
            for cell in pos_df['Cell_ID'].unique():
                cell_df = pos_df[pos_df['Cell_ID'] == cell]
                for cycle in cell_df['generation_num'].unique():
                    cycle_df =  cell_df[cell_df['generation_num'] == cycle]

                    # if (date == 20012021) & (pos == "Position_1") & (cell == 20) & (cycle == 0):
                    #     print (cycle_df)
                    frame_min = cycle_df['frame_i'].min()
                    min_frame_index = cycle_df[cycle_df['frame_i'] == frame_min].index
                    vol_min = cycle_df['cell_vol_fl'].loc[min_frame_index[0],]
                    # if (date == 20012021) & (pos == "Position_1") & (cell == 20) & (cycle == 0):
                    #     print (frame_min, min_frame_index, min_frame_index[0], vol_min)
                    frame_max = cycle_df['frame_i'].max()
                    max_frame_index = cycle_df[cycle_df['frame_i'] == frame_max].index
                    if (cycle_df['cell_cycle_stage'].loc[max_frame_index[0],] == "S"):
                        vol_max = cycle_df['sys_size_div'].loc[max_frame_index[0],]
                    else:
                        vol_max = cycle_df['cell_vol_fl'].loc[max_frame_index[0],]
                    # if (date == 20012021) & (pos == "Position_1") & (cell == 20) & (cycle == 0):
                    #     print (frame_max, max_frame_index, max_frame_index[0], vol_max)
                    delta_vol_fl = vol_max-vol_min
                    delta_time_mins = (frame_max - frame_min)*3
                    abs_growth_rate_flpermin = delta_vol_fl/delta_time_mins
                    # if (date == 20012021) & (pos == "Position_1") & (cell == 20) & (cycle == 0):
                    #     print (delta_vol_fl, delta_time_mins, abs_growth_rate_flpermin)
                    cycle_df['delta_vol_fl'] = delta_vol_fl
                    cycle_df['delta_time_mins'] = delta_time_mins
                    cycle_df['abs_growth_rate_flpermin_inclIncompleteCycles'] = abs_growth_rate_flpermin
                    if frame_max == frame_min:
                        cycle_df['delta_vol_fl'] = 0
                        cycle_df['delta_time_mins'] = 3
                        cycle_df['abs_growth_rate_flpermin_inclIncompleteCycles'] = 0/3
                    # if (date == 20012021) & (pos == "Position_1") & (cell == 20) & (cycle == 0):
                    #     print (cycle_df)
                    new.append(cycle_df)

    granddataset = pd.concat(new).reset_index(drop = True)
display (granddataset)

NameError: name 'granddataset' is not defined

# ***save new combined df***

In [11]:
granddataset.to_csv(r"Y:\dfs_for_final_code\ccr4_optimised4MLR_SCD_SCGE_merged_dataset.csv")